# Main code

In [ ]:
# ============================================================
# Notebook Notes:
# ============================================================
# This notebook implements a computationally realistic scalability
# experiment for IntervalGP-VAE.
#
# Main goal:
#   Evaluate how IntervalGP-VAE scales with increasing training
#   sample size while comparing local per-sample IntervalGP and
#   sparse inducing-point IntervalGP variants.
#
# Compared methods:
#   1) per_sample_intervalgp
#      - Uses the local per-sample IntervalGP regularizer.
#
#   2) sparse_inducing_intervalgp
#      - Uses sparse inducing-point IntervalGP regularization.
#      - Tested with inducing-point sizes:
#          m = 32
#          m = 64
#          m = 128
#
# Experiment design:
#   SEEDS = [0, 1, 2, 3, 4]
#   N_LIST = [1000, 5000, 10000]
#   M_LIST = [32, 64, 128]
#
# Debug option:
#   FAST_DEBUG = True runs a small debugging experiment:
#   SEEDS = [0]
#   N_LIST = [300]
#   M_LIST = [32]
#   TEST_SIZE = 100
#
# Synthetic data:
#   The data follow a 1D latent-confounder setting:
#      U ~ N(0, 1)
#
#   Six nonlinear proxy variables are generated from U:
#      u, u^2, u^3, sin(u), tanh(u), exp(-0.5u^2)
#
#   Treatment assignment follows:
#      T ~ Bernoulli(sigmoid(0.8U))
#
#   The structural noiseless outcome is:
#      Y(t) = 0.5U + 0.2sin(U) + t(0.8 + 0.3U)
#
#   The true ITE is computed without outcome noise.
#
# Model settings:
#   LATENT_DIM = 1
#   HIDDEN_DIM = 48
#   CAUSAL_HEAD_HIDDEN_DIM = 48
#
# GP settings:
#   GP_LENGTHSCALE = 7.0
#   GP_VARIANCE = 2.0
#   GP_NOISE = 1e-4
#   GP_JITTER = 1e-5
#
# Training settings:
#   BATCH_SIZE = 256
#   JOINT_EPOCHS = 60
#   HEAD_EPOCHS = 30
#   VAE_REFINE_EPOCHS = 10
#
#   The optimizer is AdamW with:
#      LR_JOINT = 1e-3
#      LR_HEAD = 1e-3
#      LR_VAE_REFINE = 1e-4
#      WEIGHT_DECAY = 1e-5
#
# Three-stage training framework:
#   Stage 1:
#      Jointly train the VAE and causal head.
#
#   Stage 2:
#      Freeze the VAE and train only the causal head.
#
#   Stage 3:
#      Freeze the causal head and refine the VAE.
#
# Computational realism:
#   The notebook avoids full GP inference over all training points.
#
#   Instead, it uses:
#      - reference-point subsampling
#      - sparse inducing-point regularization
#      - diagonal GP posterior prediction
#
#   MAX_GP_REF_POINTS = 128 limits the number of reference points
#   used during latent GP smoothing and ITE-space GP prediction.
#
# Inference framework:
#   1) Sample ITEs from the encoder posterior.
#   2) Apply fast latent GP posterior smoothing.
#   3) Apply fast ITE-space GP posterior prediction.
#   4) Produce ITE mean, lower bound, and upper bound.
#
# ITE uncertainty:
#   ITE_CI = 0.90
#   Z_VALUE_90 = 1.6448536269514722
#   N_ITE_SAMPLES = 30
#
# Evaluation metrics:
#   PEHE
#   ATE error
#   Interval coverage
#   Interval width
#   Raw latent correlation
#   Smoothed latent correlation
#   Absolute raw latent correlation
#   Absolute smoothed latent correlation
#
# Purpose:
#   This notebook tests whether sparse inducing-point IntervalGP
#   can preserve the accuracy and calibration behavior of the
#   per-sample IntervalGP regularizer while reducing computational
#   cost for larger training sizes.
# ============================================================

# ============================================================
# Cell 1: Imports and global settings
# ============================================================

import os
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

OUTPUT_DIR = "realistic_intervalgpvae_scalability_outputs_multi_seed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cpu"

GLOBAL_SEED = 420

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)

set_seed(GLOBAL_SEED)

torch.set_default_dtype(torch.float32)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# %%
# ============================================================
# Cell 2: Experiment configuration
# ============================================================

FAST_DEBUG = False

if FAST_DEBUG:
    SEEDS = [0]
    N_LIST = [300]
    M_LIST = [32]
    TEST_SIZE = 100
else:
    SEEDS = [0, 1, 2, 3, 4]
    N_LIST = [1000, 5000, 10000]
    M_LIST = [32, 64, 128]
    TEST_SIZE = 300

VARIANTS = [
    "per_sample_intervalgp",
    "sparse_inducing_intervalgp",
]

chosen_version = "u_aux"

LATENT_DIM = 1
HIDDEN_DIM = 48
CAUSAL_HEAD_HIDDEN_DIM = 48

GP_LENGTHSCALE = 7.0
GP_VARIANCE = 2.0
GP_NOISE = 1e-4
GP_JITTER = 1e-5

BATCH_SIZE = 256

JOINT_EPOCHS = 60
HEAD_EPOCHS = 30
VAE_REFINE_EPOCHS = 10

LR_JOINT = 1e-3
LR_HEAD = 1e-3
LR_VAE_REFINE = 1e-4
WEIGHT_DECAY = 1e-5

ITE_CI = 0.90
Z_VALUE_90 = 1.6448536269514722

N_ITE_SAMPLES = 30

MAX_GP_REF_POINTS = 128

SAVE_PARTIAL_EVERY_RUN = True


# %%
# ============================================================
# Cell 3: Utility functions
# ============================================================

def to_numpy_1d(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    return np.asarray(x).reshape(-1)


def safe_corr(x, y):
    x = to_numpy_1d(x)
    y = to_numpy_1d(y)

    valid = np.isfinite(x) & np.isfinite(y)

    if valid.sum() < 2:
        return np.nan

    if np.std(x[valid]) == 0 or np.std(y[valid]) == 0:
        return np.nan

    return float(np.corrcoef(x[valid], y[valid])[0, 1])


def pehe_np(est_ite, true_ite):
    est_ite = to_numpy_1d(est_ite)
    true_ite = to_numpy_1d(true_ite)

    valid = np.isfinite(est_ite) & np.isfinite(true_ite)

    if valid.sum() == 0:
        return np.nan

    return float(np.sqrt(np.mean((est_ite[valid] - true_ite[valid]) ** 2)))


def compute_interval_metrics(lower, upper, true_value):
    lower = to_numpy_1d(lower)
    upper = to_numpy_1d(upper)
    true_value = to_numpy_1d(true_value)

    valid = (
        np.isfinite(lower)
        & np.isfinite(upper)
        & np.isfinite(true_value)
    )

    if valid.sum() == 0:
        return {
            "coverage": np.nan,
            "avg_length": np.nan,
            "n": 0,
        }

    covered = (
        (lower[valid] <= true_value[valid])
        & (true_value[valid] <= upper[valid])
    )

    return {
        "coverage": float(np.mean(covered)),
        "avg_length": float(np.mean(upper[valid] - lower[valid])),
        "n": int(valid.sum()),
    }


def reset_memory():
    gc.collect()


def choose_random_subset(n, max_points, seed=0):
    if max_points is None or n <= max_points:
        return np.arange(n)

    rng = np.random.default_rng(seed)
    idx = rng.choice(np.arange(n), size=max_points, replace=False)
    return np.sort(idx)


# %%
# ============================================================
# Cell 4: Synthetic proxy-based causal data
# ============================================================

def proxy_functions_1d(u):
    """
    u: (n, 1)
    return z: (n, 6)
    """
    return torch.cat(
        [
            u,
            u ** 2,
            u ** 3,
            torch.sin(u),
            torch.tanh(u),
            torch.exp(-0.5 * u ** 2),
        ],
        dim=1,
    )


def treatment_func(u_np):
    """
    T ~ Bernoulli(sigmoid(0.8U)).
    """
    logits = 0.8 * u_np
    probs = 1.0 / (1.0 + np.exp(-logits))
    return np.random.binomial(1, probs).astype(np.float32).reshape(-1)


def outcome_noiseless(u_np, t_np):
    """
    Structural noiseless outcome:
        Y(t) = 0.5U + 0.2sin(U) + t(0.8 + 0.3U)
    """
    if t_np.ndim > 1:
        t_np = t_np.reshape(-1)

    return (
        0.5 * u_np
        + 0.2 * np.sin(u_np)
        + t_np.reshape(-1, 1) * (0.8 + 0.3 * u_np)
    ).astype(np.float32)


def generate_synthetic_data(
    n,
    seed,
    noise_z=0.1,
    noise_y=0.1,
):
    set_seed(seed)

    u_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)
    u_tensor = torch.from_numpy(u_np)

    clean_z = proxy_functions_1d(u_tensor)
    z = clean_z + noise_z * torch.randn_like(clean_z)

    t_np = treatment_func(u_np)

    y_clean = outcome_noiseless(u_np, t_np)
    y_np = y_clean + np.random.normal(
        0.0,
        noise_y,
        size=y_clean.shape,
    ).astype(np.float32)

    t0_np = np.zeros(n, dtype=np.float32)
    t1_np = np.ones(n, dtype=np.float32)

    true_ite = outcome_noiseless(u_np, t1_np) - outcome_noiseless(u_np, t0_np)

    return {
        "z": z.float(),
        "t": torch.from_numpy(t_np).float(),
        "y": torch.from_numpy(y_np.squeeze()).float(),
        "u": torch.from_numpy(u_np.squeeze()).float(),
        "true_ite": torch.from_numpy(true_ite.squeeze()).float(),
    }


# %%
# ============================================================
# Cell 5: Fast kernel and subset GP posterior
# ============================================================

def rbf_kernel(
    x1,
    x2=None,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
):
    if x2 is None:
        x2 = x1

    if x1.ndim == 1:
        x1 = x1.view(-1, 1)

    if x2.ndim == 1:
        x2 = x2.view(-1, 1)

    x1 = x1.float()
    x2 = x2.float()

    dists = torch.cdist(x1, x2).pow(2)

    return variance * torch.exp(-0.5 * dists / (lengthscale ** 2))


def safe_cholesky(K, jitter=GP_JITTER, max_tries=8):
    eye = torch.eye(K.shape[0], device=K.device, dtype=K.dtype)
    current_jitter = jitter

    for _ in range(max_tries):
        try:
            return torch.linalg.cholesky(K + current_jitter * eye)
        except RuntimeError:
            current_jitter *= 10.0

    return torch.linalg.cholesky(K + current_jitter * eye)


def inducing_gp_predict_diag(
    x_ref,
    y_ref,
    x_test,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
    jitter=GP_JITTER,
):
    """
    Fast GP posterior using only reference points.

    Returns:
        posterior_mean: (N_test,)
        posterior_std:  (N_test,)

    No full N_test x N_test covariance matrix is constructed.
    """
    if x_ref.ndim == 1:
        x_ref = x_ref.view(-1, 1)

    if x_test.ndim == 1:
        x_test = x_test.view(-1, 1)

    y_ref = y_ref.float().view(-1, 1)

    M = x_ref.shape[0]

    K_mm = rbf_kernel(
        x_ref,
        x_ref,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_mm = K_mm + noise * torch.eye(M, device=x_ref.device)

    K_mn = rbf_kernel(
        x_ref,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    L = safe_cholesky(K_mm, jitter=jitter)

    alpha = torch.cholesky_solve(y_ref, L)
    mean = K_mn.t() @ alpha
    mean = mean.squeeze(-1)

    v = torch.cholesky_solve(K_mn, L)
    var_diag = variance - torch.sum(K_mn * v, dim=0)
    var_diag = torch.clamp(var_diag, min=1e-8)

    std = torch.sqrt(var_diag)

    return mean, std


def select_reference_points(x, *ys, max_points=MAX_GP_REF_POINTS, seed=0):
    n = x.shape[0]
    idx_np = choose_random_subset(n, max_points=max_points, seed=seed)
    idx = torch.tensor(idx_np, dtype=torch.long, device=x.device)

    out = [x[idx]]

    for y in ys:
        out.append(y[idx])

    return tuple(out)


# %%
# ============================================================
# Cell 6: Training-time IntervalGP regularizers
# ============================================================

def local_interval_gp_regularizer(
    mu_u,
    std_u,
    gp_variance=GP_VARIANCE,
    gp_noise=GP_NOISE,
):
    """
    Per-sample / local IntervalGP regularizer.
    """
    k_ii = gp_variance * torch.ones_like(mu_u)
    gp_std = torch.sqrt(k_ii + gp_noise)

    lower = mu_u - std_u
    upper = mu_u + std_u

    normal = torch.distributions.Normal(
        loc=torch.zeros_like(mu_u),
        scale=gp_std,
    )

    prob = normal.cdf(upper) - normal.cdf(lower)
    prob = torch.clamp(prob, min=1e-8)

    return -torch.mean(torch.log(prob))


def sparse_inducing_intervalgp_regularizer(
    z_batch,
    mu_u,
    std_u,
    inducing_z,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
    jitter=GP_JITTER,
):
    """
    Sparse inducing-point IntervalGP regularizer.
    """
    B = z_batch.shape[0]
    M = inducing_z.shape[0]
    device = z_batch.device

    K_mm = rbf_kernel(
        inducing_z,
        inducing_z,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_mm = K_mm + jitter * torch.eye(M, device=device)

    K_nm = rbf_kernel(
        z_batch,
        inducing_z,
        lengthscale=lengthscale,
        variance=variance,
    )

    L_mm = safe_cholesky(K_mm, jitter=jitter)

    v = torch.linalg.solve_triangular(
        L_mm,
        K_nm.t(),
        upper=False,
    )

    approx_var = torch.sum(v ** 2, dim=0)
    approx_var = torch.clamp(approx_var, min=noise, max=variance + noise)

    scale = torch.sqrt(approx_var + noise)
    scale = scale.view(B, 1).expand_as(mu_u)

    normal = torch.distributions.Normal(
        loc=torch.zeros_like(mu_u),
        scale=scale,
    )

    lower = mu_u - std_u
    upper = mu_u + std_u

    prob = normal.cdf(upper) - normal.cdf(lower)
    prob = torch.clamp(prob, min=1e-8)

    return -torch.mean(torch.log(prob))


# %%
# ============================================================
# Cell 7: Model definition
# ============================================================

class DoubleEncoder(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.fc1 = nn.Linear(input_dim, hidden_dim)

        self.mean_u = nn.Linear(hidden_dim, latent_dim)
        self.logvar_u = nn.Linear(hidden_dim, latent_dim)

        self.mean_eps = nn.Linear(hidden_dim, input_dim)
        self.logvar_eps = nn.Linear(hidden_dim, input_dim)

        if self.use_aux:
            self.mean_ua0 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua0 = nn.Linear(hidden_dim, latent_dim)

            self.mean_ua1 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua1 = nn.Linear(hidden_dim, latent_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))

        mu_u = self.mean_u(h)
        logvar_u = self.logvar_u(h).clamp(-8.0, 4.0)
        std_u = torch.exp(0.5 * logvar_u)

        mu_eps = self.mean_eps(h)
        logvar_eps = self.logvar_eps(h).clamp(-8.0, 4.0)
        std_eps = torch.exp(0.5 * logvar_eps)

        if self.use_aux:
            mu_ua0 = self.mean_ua0(h)
            logvar_ua0 = self.logvar_ua0(h).clamp(-8.0, 4.0)
            std_ua0 = torch.exp(0.5 * logvar_ua0)

            mu_ua1 = self.mean_ua1(h)
            logvar_ua1 = self.logvar_ua1(h).clamp(-8.0, 4.0)
            std_ua1 = torch.exp(0.5 * logvar_ua1)

            return (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
                mu_ua0,
                std_ua0,
                logvar_ua0,
                mu_ua1,
                std_ua1,
                logvar_ua1,
            )

        return mu_u, std_u, logvar_u, mu_eps, std_eps, logvar_eps


class AdditiveDecoder(nn.Module):
    def __init__(
        self,
        latent_dim,
        hidden_dim,
        output_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents
        factor = 1 + (2 if use_auxiliary_latents else 0)

        self.fc1 = nn.Linear(latent_dim * factor, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, u, eps, ua0=None, ua1=None):
        if self.use_aux:
            x = torch.cat([u, ua0, ua1], dim=1)
        else:
            x = u

        h = F.relu(self.fc1(x))
        return self.out(h) + eps


class FlexibleCausalHead(nn.Module):
    def __init__(
        self,
        latent_dim_u,
        hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
        z_y_dim=None,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents
        self.z_y_dim = z_y_dim

        input_dim = latent_dim_u + 1

        if z_y_dim is not None:
            input_dim += z_y_dim

        if use_auxiliary_latents:
            input_dim += latent_dim_u

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, u, t, z_y=None, ua1=None):
        t = t.view(-1, 1)

        parts = [u, t]

        if self.z_y_dim is not None:
            if z_y is None:
                raise ValueError("z_y is required but got None.")
            parts.append(z_y)

        if self.use_aux:
            if ua1 is None:
                raise ValueError("ua1 is required but got None.")
            parts.append(ua1)

        x = torch.cat(parts, dim=1)
        h = F.relu(self.fc1(x))

        return self.out(h)


def get_z_y_dim(chosen_version, z_dim):
    if chosen_version == "z_to_y":
        return z_dim

    if chosen_version == "split_z_to_t_and_y":
        return z_dim // 2

    return None


def get_use_aux(chosen_version):
    return chosen_version == "u_aux"


def make_z_y(model, z):
    if model.causal_head.z_y_dim == z.shape[1]:
        return z

    if model.causal_head.z_y_dim:
        return z[:, z.shape[1] // 2:]

    return None


class GPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        gp_variant="per_sample_intervalgp",
        inducing_z=None,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.encoder = DoubleEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.decoder = AdditiveDecoder(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            output_dim=input_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.gp_variant = gp_variant
        self.gp_lengthscale = gp_lengthscale
        self.gp_variance = gp_variance
        self.gp_noise = gp_noise

        if inducing_z is not None:
            self.register_buffer("inducing_z", inducing_z.clone().float())
        else:
            self.inducing_z = None

    def reparameterize(self, mu, std):
        return mu + torch.randn_like(std) * std

    def get_latent_stats(self, z, var_name="u"):
        outputs = self.encoder(z)

        if var_name == "u":
            return outputs[0], outputs[1], outputs[2]

        if var_name == "eps":
            return outputs[3], outputs[4], outputs[5]

        if var_name == "ua0":
            if not self.use_aux:
                return None, None, None
            return outputs[6], outputs[7], outputs[8]

        if var_name == "ua1":
            if not self.use_aux:
                return None, None, None
            return outputs[9], outputs[10], outputs[11]

        raise ValueError(f"Invalid var_name: {var_name}")

    def sample_latent(self, z, var_name="u"):
        mu, std, _ = self.get_latent_stats(z, var_name=var_name)

        if mu is None:
            return None

        return self.reparameterize(mu, std)

    def gp_regularizer(self, z, mu_u, std_u):
        if self.gp_variant == "per_sample_intervalgp":
            return local_interval_gp_regularizer(
                mu_u=mu_u,
                std_u=std_u,
                gp_variance=self.gp_variance,
                gp_noise=self.gp_noise,
            )

        if self.gp_variant == "sparse_inducing_intervalgp":
            if self.inducing_z is None:
                raise ValueError("Sparse inducing IntervalGP requires inducing_z.")

            return sparse_inducing_intervalgp_regularizer(
                z_batch=z,
                mu_u=mu_u,
                std_u=std_u,
                inducing_z=self.inducing_z,
                lengthscale=self.gp_lengthscale,
                variance=self.gp_variance,
                noise=self.gp_noise,
            )

        raise ValueError(f"Unknown gp_variant: {self.gp_variant}")

    def forward(self, z):
        if self.use_aux:
            (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
                mu_ua0,
                std_ua0,
                logvar_ua0,
                mu_ua1,
                std_ua1,
                logvar_ua1,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = self.reparameterize(mu_ua0, std_ua0)
            ua1 = self.reparameterize(mu_ua1, std_ua1)

            z_recon = self.decoder(u, eps, ua0=ua0, ua1=ua1)

        else:
            (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = None
            ua1 = None

            z_recon = self.decoder(u, eps)

        recon_loss = F.mse_loss(z_recon, z, reduction="mean")

        kl_u = -0.001 * torch.mean(
            1.0 + logvar_u - mu_u.pow(2) - logvar_u.exp()
        )

        kl_eps = -0.5 * torch.mean(
            1.0 + logvar_eps - mu_eps.pow(2) - logvar_eps.exp()
        )

        gp_interval_reg = self.gp_regularizer(
            z=z,
            mu_u=mu_u,
            std_u=std_u,
        )

        std_penalty = torch.mean(std_u ** 2)

        total_loss = (
            recon_loss
            + kl_u
            + kl_eps
            + gp_interval_reg
            + 1.0 * std_penalty
        )

        if self.use_aux:
            kl_ua0 = -0.5 * torch.mean(
                1.0 + logvar_ua0 - mu_ua0.pow(2) - logvar_ua0.exp()
            )

            kl_ua1 = -0.5 * torch.mean(
                1.0 + logvar_ua1 - mu_ua1.pow(2) - logvar_ua1.exp()
            )

            total_loss = total_loss + kl_ua0 + kl_ua1

        return total_loss, {
            "u": u,
            "eps": eps,
            "ua0": ua0,
            "ua1": ua1,
            "mu_u": mu_u,
            "std_u": std_u,
            "z_recon": z_recon,
        }


class CausalGPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        z_y_dim=None,
        use_auxiliary_latents=False,
        gp_variant="per_sample_intervalgp",
        inducing_z=None,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
    ):
        super().__init__()

        self.z_y_dim = z_y_dim
        self.use_aux = use_auxiliary_latents

        self.vae = GPVAEwithNoise(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            gp_variant=gp_variant,
            inducing_z=inducing_z,
            gp_lengthscale=gp_lengthscale,
            gp_variance=gp_variance,
            gp_noise=gp_noise,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.causal_head = FlexibleCausalHead(
            latent_dim_u=latent_dim,
            hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
            z_y_dim=z_y_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

    def forward(self, z, t, y):
        loss_vae, vae_info = self.vae(z)

        u = vae_info["u"]
        ua1 = vae_info["ua1"] if self.use_aux else None

        z_y = make_z_y(self, z)

        y_pred = self.causal_head(
            u,
            t,
            z_y=z_y,
            ua1=ua1,
        )

        causal_loss = F.mse_loss(
            y_pred,
            y.unsqueeze(-1),
            reduction="mean",
        )

        total_loss = loss_vae + causal_loss

        return total_loss, {
            **vae_info,
            "y_pred": y_pred,
        }


# %%
# ============================================================
# Cell 8: Inducing point selection
# ============================================================

def select_inducing_points_random(z_train, m, seed):
    set_seed(seed)

    n = z_train.shape[0]
    m_eff = min(m, n)

    idx = torch.randperm(n)[:m_eff]

    return z_train[idx].clone()


# %%
# ============================================================
# Cell 9: Fast ITE inference
# ============================================================

def sample_ite_from_encoder_fast(
    model,
    z,
    n_samples=N_ITE_SAMPLES,
):
    model.eval()

    with torch.no_grad():
        mu_u, std_u, _ = model.vae.get_latent_stats(z, var_name="u")
        N, d_u = mu_u.shape
        device = z.device

        z_y = make_z_y(model, z)

        mu_ua1, std_ua1, _ = model.vae.get_latent_stats(z, var_name="ua1")
        has_ua1 = (mu_ua1 is not None) and (std_ua1 is not None)

        eps_u = torch.randn(n_samples, N, d_u, device=device)
        u_samples = mu_u.unsqueeze(0) + eps_u * std_u.unsqueeze(0)
        U = u_samples.reshape(-1, d_u)

        if has_ua1:
            d_ua = mu_ua1.shape[1]
            eps_ua1 = torch.randn(n_samples, N, d_ua, device=device)
            ua1_samples = mu_ua1.unsqueeze(0) + eps_ua1 * std_ua1.unsqueeze(0)
            UA1 = ua1_samples.reshape(-1, d_ua)
        else:
            UA1 = None

        if z_y is not None:
            ZY = (
                z_y.unsqueeze(0)
                .expand(n_samples, -1, -1)
                .reshape(-1, z_y.shape[1])
            )
        else:
            ZY = None

        t0 = torch.zeros(U.shape[0], device=device)
        t1 = torch.ones(U.shape[0], device=device)

        y0 = model.causal_head(U, t0, z_y=ZY, ua1=UA1).squeeze(-1)
        y1 = model.causal_head(U, t1, z_y=ZY, ua1=UA1).squeeze(-1)

        ite_samples = (y1 - y0).view(n_samples, N)

        lower_q = (1.0 - ITE_CI) / 2.0
        upper_q = 1.0 - lower_q

        ite_mean = ite_samples.mean(dim=0)
        ite_std = ite_samples.std(dim=0)
        ite_lower = ite_samples.quantile(lower_q, dim=0)
        ite_upper = ite_samples.quantile(upper_q, dim=0)

        return ite_samples, ite_mean, ite_std, ite_lower, ite_upper


def fast_latent_gp_smoothing(
    z_train,
    mu_u_train,
    z_test,
    max_ref_points=MAX_GP_REF_POINTS,
    seed=0,
):
    if mu_u_train.ndim == 1:
        mu_u_train = mu_u_train.view(-1, 1)

    refs = select_reference_points(
        z_train,
        mu_u_train,
        max_points=max_ref_points,
        seed=seed,
    )

    z_ref = refs[0]
    mu_u_ref = refs[1]

    latent_dim = mu_u_train.shape[1]

    means = []
    stds = []

    for j in range(latent_dim):
        mean_j, std_j = inducing_gp_predict_diag(
            x_ref=z_ref,
            y_ref=mu_u_ref[:, j],
            x_test=z_test,
            lengthscale=GP_LENGTHSCALE,
            variance=GP_VARIANCE,
            noise=GP_NOISE,
        )

        means.append(mean_j)
        stds.append(std_j)

    mean = torch.stack(means, dim=1)
    std = torch.stack(stds, dim=1)

    return mean, std


def fast_ite_space_intervalgp_head(
    model,
    z_train,
    z_test,
    n_samples=N_ITE_SAMPLES,
    max_ref_points=MAX_GP_REF_POINTS,
    seed=0,
):
    model.eval()

    with torch.no_grad():
        _, ite_mean_train, ite_std_train, ite_lower_train, ite_upper_train = (
            sample_ite_from_encoder_fast(
                model=model,
                z=z_train,
                n_samples=n_samples,
            )
        )

        mu_u_train, _, _ = model.vae.get_latent_stats(z_train, var_name="u")
        mu_u_test, _, _ = model.vae.get_latent_stats(z_test, var_name="u")

        u_test_smooth, u_test_smooth_std = fast_latent_gp_smoothing(
            z_train=z_train,
            mu_u_train=mu_u_train,
            z_test=z_test,
            max_ref_points=max_ref_points,
            seed=seed + 17,
        )

        refs = select_reference_points(
            mu_u_train.detach(),
            ite_mean_train.detach(),
            ite_lower_train.detach(),
            ite_upper_train.detach(),
            ite_std_train.detach(),
            max_points=max_ref_points,
            seed=seed + 29,
        )

        u_ref = refs[0]
        ite_mean_ref = refs[1]
        ite_lower_ref = refs[2]
        ite_upper_ref = refs[3]
        ite_std_ref = refs[4]

        effective_noise = GP_NOISE + torch.mean(ite_std_ref ** 2).item()

        gp_ite_mean, gp_ite_std = inducing_gp_predict_diag(
            x_ref=u_ref,
            y_ref=ite_mean_ref,
            x_test=u_test_smooth,
            lengthscale=GP_LENGTHSCALE,
            variance=GP_VARIANCE,
            noise=effective_noise,
        )

        gp_lower_mean, gp_lower_std = inducing_gp_predict_diag(
            x_ref=u_ref,
            y_ref=ite_lower_ref,
            x_test=u_test_smooth,
            lengthscale=GP_LENGTHSCALE,
            variance=GP_VARIANCE,
            noise=effective_noise,
        )

        gp_upper_mean, gp_upper_std = inducing_gp_predict_diag(
            x_ref=u_ref,
            y_ref=ite_upper_ref,
            x_test=u_test_smooth,
            lengthscale=GP_LENGTHSCALE,
            variance=GP_VARIANCE,
            noise=effective_noise,
        )

        raw_lower = torch.minimum(gp_lower_mean, gp_upper_mean)
        raw_upper = torch.maximum(gp_lower_mean, gp_upper_mean)

        ite_lower_test = raw_lower - Z_VALUE_90 * gp_lower_std
        ite_upper_test = raw_upper + Z_VALUE_90 * gp_upper_std

        return {
            "ite_mean_test": gp_ite_mean,
            "ite_std_test": gp_ite_std,
            "ite_lower_test": ite_lower_test,
            "ite_upper_test": ite_upper_test,
            "u_test_smooth": u_test_smooth,
            "u_test_smooth_std": u_test_smooth_std,
            "u_test_raw": mu_u_test,
            "ite_lower_train": ite_lower_train,
            "ite_upper_train": ite_upper_train,
            "ite_mean_train": ite_mean_train,
            "ite_std_train": ite_std_train,
        }


# %%
# ============================================================
# Cell 10: Three-stage training
# ============================================================

def train_intervalgpvae_three_stage(
    model,
    loader,
    device=DEVICE,
    verbose=False,
):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR_JOINT,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(JOINT_EPOCHS):
        model.train()

        for z_batch, t_batch, y_batch in loader:
            z_batch = z_batch.to(device)
            t_batch = t_batch.to(device)
            y_batch = y_batch.to(device)

            loss, _ = model(z_batch, t_batch, y_batch)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

    for param in model.vae.parameters():
        param.requires_grad = False

    for param in model.causal_head.parameters():
        param.requires_grad = True

    causal_optimizer = torch.optim.AdamW(
        model.causal_head.parameters(),
        lr=LR_HEAD,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(HEAD_EPOCHS):
        model.train()

        for z_batch, t_batch, y_batch in loader:
            z_batch = z_batch.to(device)
            t_batch = t_batch.to(device)
            y_batch = y_batch.to(device)

            with torch.no_grad():
                mu_u = model.vae.get_latent_stats(z_batch, var_name="u")[0]

                if model.use_aux:
                    mu_ua1, _, _ = model.vae.get_latent_stats(
                        z_batch,
                        var_name="ua1",
                    )
                    ua1_batch = mu_ua1
                else:
                    ua1_batch = None

            z_y_batch = make_z_y(model, z_batch)

            y_pred = model.causal_head(
                mu_u.detach(),
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch.detach() if ua1_batch is not None else None,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
                reduction="mean",
            )

            causal_optimizer.zero_grad()
            causal_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.causal_head.parameters(), max_norm=5.0)
            causal_optimizer.step()

    for param in model.causal_head.parameters():
        param.requires_grad = False

    for param in model.vae.parameters():
        param.requires_grad = True

    vae_optimizer = torch.optim.AdamW(
        model.vae.parameters(),
        lr=LR_VAE_REFINE,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(VAE_REFINE_EPOCHS):
        model.train()

        for z_batch, t_batch, y_batch in loader:
            z_batch = z_batch.to(device)
            t_batch = t_batch.to(device)
            y_batch = y_batch.to(device)

            loss_vae, vae_info = model.vae(z_batch)

            z_y_batch = make_z_y(model, z_batch)

            if model.use_aux:
                ua1_batch = vae_info["ua1"]
            else:
                ua1_batch = None

            y_pred = model.causal_head(
                vae_info["u"],
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
                reduction="mean",
            )

            total_loss = loss_vae + causal_loss

            vae_optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.vae.parameters(), max_norm=5.0)
            vae_optimizer.step()

    for param in model.parameters():
        param.requires_grad = True

    return model


# %%
# ============================================================
# Cell 11: One-run training and evaluation
# ============================================================

def train_one_model(
    data_train,
    data_test,
    variant,
    m=None,
    seed=0,
    device=DEVICE,
):
    set_seed(seed)
    reset_memory()

    z_train = data_train["z"].to(device)
    t_train = data_train["t"].to(device)
    y_train = data_train["y"].to(device)

    z_test = data_test["z"].to(device)
    true_ite_test = data_test["true_ite"].to(device)
    true_u_test = data_test["u"].to(device)

    z_mean = z_train.mean(dim=0, keepdim=True)
    z_std = z_train.std(dim=0, keepdim=True).clamp_min(1e-6)

    z_train = (z_train - z_mean) / z_std
    z_test = (z_test - z_mean) / z_std

    inducing_z = None

    if variant == "sparse_inducing_intervalgp":
        inducing_z = select_inducing_points_random(
            z_train=z_train.detach(),
            m=m,
            seed=seed,
        ).to(device)

    z_y_dim = get_z_y_dim(chosen_version, z_train.shape[1])
    use_aux = get_use_aux(chosen_version)

    model = CausalGPVAEwithNoise(
        input_dim=z_train.shape[1],
        hidden_dim=HIDDEN_DIM,
        latent_dim=LATENT_DIM,
        z_y_dim=z_y_dim,
        use_auxiliary_latents=use_aux,
        gp_variant=variant,
        inducing_z=inducing_z,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
    ).to(device)

    loader = DataLoader(
        TensorDataset(z_train, t_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
    )

    start_time = time.time()

    model = train_intervalgpvae_three_stage(
        model=model,
        loader=loader,
        device=device,
        verbose=False,
    )

    training_time = time.time() - start_time

    eval_start_time = time.time()

    results = fast_ite_space_intervalgp_head(
        model=model,
        z_train=z_train,
        z_test=z_test,
        n_samples=N_ITE_SAMPLES,
        max_ref_points=MAX_GP_REF_POINTS,
        seed=seed,
    )

    eval_time = time.time() - eval_start_time

    ite_mean = results["ite_mean_test"]
    ite_lower = results["ite_lower_test"]
    ite_upper = results["ite_upper_test"]

    pehe = pehe_np(ite_mean, true_ite_test)

    ate_est = float(to_numpy_1d(ite_mean).mean())
    ate_true = float(to_numpy_1d(true_ite_test).mean())
    ate_error = abs(ate_est - ate_true)

    interval_info = compute_interval_metrics(
        lower=ite_lower,
        upper=ite_upper,
        true_value=true_ite_test,
    )

    raw_u_corr = safe_corr(results["u_test_raw"], true_u_test)
    smooth_u_corr = safe_corr(results["u_test_smooth"], true_u_test)

    metrics = {
        "PEHE": pehe,
        "ATE_error": ate_error,
        "coverage": interval_info["coverage"],
        "interval_width": interval_info["avg_length"],
        "ATE_est": ate_est,
        "ATE_true": ate_true,
        "raw_u_corr": raw_u_corr,
        "smooth_u_corr": smooth_u_corr,
        "abs_raw_u_corr": abs(raw_u_corr) if np.isfinite(raw_u_corr) else np.nan,
        "abs_smooth_u_corr": abs(smooth_u_corr) if np.isfinite(smooth_u_corr) else np.nan,
        "training_time_sec": training_time,
        "eval_time_sec": eval_time,
        "total_time_sec": training_time + eval_time,
    }

    del model
    gc.collect()

    return metrics


# %%
# ============================================================
# Cell 12: Main experiment loop
# ============================================================

all_results = []

total_runs = 0

for n_train in N_LIST:
    for seed in SEEDS:
        for variant in VARIANTS:
            if variant == "sparse_inducing_intervalgp":
                total_runs += len(M_LIST)
            else:
                total_runs += 1

print(f"Total planned runs: {total_runs}")

run_id = 0

for n_train in N_LIST:
    for seed in SEEDS:
        print("\n" + "=" * 100)
        print(f"Generating data: n_train={n_train}, seed={seed}")
        print("=" * 100)

        data_train = generate_synthetic_data(
            n=n_train,
            seed=1000 + seed,
        )

        data_test = generate_synthetic_data(
            n=TEST_SIZE,
            seed=9000 + seed,
        )

        for variant in VARIANTS:
            if variant == "sparse_inducing_intervalgp":
                m_values = M_LIST
            else:
                m_values = [None]

            for m in m_values:
                run_id += 1

                print("\n" + "-" * 100)
                print(
                    f"Run {run_id}/{total_runs} | "
                    f"variant={variant}, m={m}, n_train={n_train}, seed={seed}"
                )
                print("-" * 100)

                try:
                    metrics = train_one_model(
                        data_train=data_train,
                        data_test=data_test,
                        variant=variant,
                        m=m,
                        seed=seed,
                        device=DEVICE,
                    )

                    row = {
                        "run_id": run_id,
                        "n_train": n_train,
                        "test_size": TEST_SIZE,
                        "seed": seed,
                        "variant": variant,
                        "m": m,
                        "device": DEVICE,
                        "chosen_version": chosen_version,
                        "latent_dim": LATENT_DIM,
                        "hidden_dim": HIDDEN_DIM,
                        "causal_head_hidden_dim": CAUSAL_HEAD_HIDDEN_DIM,
                        "gp_lengthscale": GP_LENGTHSCALE,
                        "gp_variance": GP_VARIANCE,
                        "gp_noise": GP_NOISE,
                        "max_gp_ref_points": MAX_GP_REF_POINTS,
                        "batch_size": BATCH_SIZE,
                        "joint_epochs": JOINT_EPOCHS,
                        "head_epochs": HEAD_EPOCHS,
                        "vae_refine_epochs": VAE_REFINE_EPOCHS,
                        "n_ite_samples": N_ITE_SAMPLES,
                        **metrics,
                    }

                    all_results.append(row)

                    print(
                        f"Done: PEHE={metrics['PEHE']:.4f}, "
                        f"ATE error={metrics['ATE_error']:.4f}, "
                        f"coverage={metrics['coverage']:.4f}, "
                        f"width={metrics['interval_width']:.4f}, "
                        f"train={metrics['training_time_sec']:.2f}s, "
                        f"eval={metrics['eval_time_sec']:.2f}s, "
                        f"total={metrics['total_time_sec']:.2f}s"
                    )

                except Exception as e:
                    print(
                        f"Error for variant={variant}, m={m}, "
                        f"n_train={n_train}, seed={seed}"
                    )
                    print(e)

                    row = {
                        "run_id": run_id,
                        "n_train": n_train,
                        "test_size": TEST_SIZE,
                        "seed": seed,
                        "variant": variant,
                        "m": m,
                        "device": DEVICE,
                        "error": str(e),
                    }

                    all_results.append(row)

                if SAVE_PARTIAL_EVERY_RUN:
                    partial_df = pd.DataFrame(all_results)
                    partial_path = os.path.join(
                        OUTPUT_DIR,
                        "realistic_intervalgpvae_partial_results.csv",
                    )
                    partial_df.to_csv(partial_path, index=False)
                    print(f"Partial results saved to: {partial_path}")


# %%
# ============================================================
# Cell 13: Save final results
# ============================================================

results_df = pd.DataFrame(all_results)

results_path = os.path.join(
    OUTPUT_DIR,
    "realistic_intervalgpvae_results.csv",
)

results_df.to_csv(results_path, index=False)

print("\n" + "=" * 100)
print(f"Full results saved to: {results_path}")
print("=" * 100)

display(results_df)


# %%
# ============================================================
# Cell 14: Summary table
# ============================================================

valid_results_df = results_df[results_df["PEHE"].notna()].copy()

summary_df = (
    valid_results_df
    .groupby(["n_train", "variant", "m"], dropna=False)
    .agg(
        num_runs=("PEHE", "count"),
        PEHE_mean=("PEHE", "mean"),
        PEHE_std=("PEHE", "std"),
        ATE_error_mean=("ATE_error", "mean"),
        ATE_error_std=("ATE_error", "std"),
        coverage_mean=("coverage", "mean"),
        coverage_std=("coverage", "std"),
        interval_width_mean=("interval_width", "mean"),
        interval_width_std=("interval_width", "std"),
        training_time_mean=("training_time_sec", "mean"),
        training_time_std=("training_time_sec", "std"),
        eval_time_mean=("eval_time_sec", "mean"),
        eval_time_std=("eval_time_sec", "std"),
        total_time_mean=("total_time_sec", "mean"),
        total_time_std=("total_time_sec", "std"),
        raw_u_corr_mean=("raw_u_corr", "mean"),
        raw_u_corr_std=("raw_u_corr", "std"),
        smooth_u_corr_mean=("smooth_u_corr", "mean"),
        smooth_u_corr_std=("smooth_u_corr", "std"),
        abs_raw_u_corr_mean=("abs_raw_u_corr", "mean"),
        abs_raw_u_corr_std=("abs_raw_u_corr", "std"),
        abs_smooth_u_corr_mean=("abs_smooth_u_corr", "mean"),
        abs_smooth_u_corr_std=("abs_smooth_u_corr", "std"),
    )
    .reset_index()
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "realistic_intervalgpvae_summary.csv",
)

summary_df.to_csv(summary_path, index=False)

print("\nSummary:")
display(summary_df)

print(f"\nSummary saved to: {summary_path}")


# %%
# ============================================================
# Cell 15: Compact paper-style summary
# ============================================================

def method_label(row):
    variant = row["variant"]
    m = row["m"]

    if variant == "per_sample_intervalgp":
        return "Per-sample"

    if variant == "sparse_inducing_intervalgp":
        return f"Sparse-{int(m)}"

    return str(variant)


summary_df["method"] = summary_df.apply(method_label, axis=1)

compact_summary = summary_df[
    [
        "n_train",
        "method",
        "num_runs",
        "PEHE_mean",
        "PEHE_std",
        "ATE_error_mean",
        "ATE_error_std",
        "coverage_mean",
        "coverage_std",
        "interval_width_mean",
        "interval_width_std",
        "training_time_mean",
        "training_time_std",
        "eval_time_mean",
        "eval_time_std",
        "total_time_mean",
        "total_time_std",
        "abs_raw_u_corr_mean",
        "abs_raw_u_corr_std",
        "abs_smooth_u_corr_mean",
        "abs_smooth_u_corr_std",
    ]
].copy()

compact_summary_path = os.path.join(
    OUTPUT_DIR,
    "realistic_intervalgpvae_compact_summary.csv",
)

compact_summary.to_csv(compact_summary_path, index=False)

print("\nCompact summary:")
display(compact_summary)

print(f"\nCompact summary saved to: {compact_summary_path}")


# %%
# ============================================================
# Cell 16: Optional quick ranking table
# ============================================================

ranking_df = compact_summary.sort_values(
    by=["n_train", "PEHE_mean", "coverage_mean"],
    ascending=[True, True, False],
).copy()

ranking_path = os.path.join(
    OUTPUT_DIR,
    "realistic_intervalgpvae_ranking.csv",
)

ranking_df.to_csv(ranking_path, index=False)

print("\nRanking table:")
display(ranking_df)

print(f"\nRanking saved to: {ranking_path}")

# End